In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
from huggingface_hub import login

/tmp/ipykernel_571/951077658.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [ ]:
# model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Using device:", device)

# tokenizer = AutoTokenizer.from_pretrained(model_name)

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.float16,
#     device_map="gpu"
# )

# # model = AutoModelForCausalLM.from_pretrained(
# #     model_name,
# #     dtype=torch.float16
# # )


# if not torch.cuda.is_available():
#     model.to(device)

# print("Model device:", next(model.parameters()).device)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
)

model.to(device)

print("Model device:", next(model.parameters()).device)

Using device: cuda


model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model device: cuda:0


In [ ]:
phrase = input("Write a phrase: ")
inputs = tokenizer(phrase, return_tensors = "pt")
inputs = {k: v.to(device) for k, v in inputs.items()}
inputs

Write a phrase: Once upon a time there was a little boy


{'input_ids': tensor([[6403, 1980,  253,  655,  665,  436,  253, 1838, 7706]],
        device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

Once upon a time there was a little boy who lived in a small village. He was very poor and had to work hard every day to make ends meet. He had to help his father in the fields and his mother in the house. He was very tired and hungry all the time. One day, he saw a beautiful palace in the distance. He ran towards it and knocked on the door. A beautiful lady opened the door and asked him who he was and what he wanted. The little boy told her that he was a poor boy who lived in the village and that he had come to see the palace. The lady was very kind and took him in. She fed him and gave him a warm bed to sleep in. She told him that she was the queen of the palace and that she was very kind and generous. She asked him if he wanted to stay with her and her family. The little boy was very happy and said yes. He stayed with the queen and her family for many years. He was very happy and had a


In [ ]:
# Create a Hugging Face text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    return_full_text=False
)

# Wrap it for LangChain
llm = HuggingFacePipeline(pipeline=pipe)

# Wrap it AGAIN to enable Chat and Tool Calling formats
chat_model = ChatHuggingFace(llm=llm)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
@tool
def search_local_database(query: str) -> str:
    """Use this tool to search the local vector database for context."""
    # (Your ChromaDB or FAISS retrieval code will go here later)
    return f"Retrieved documents about: {query}"

# Bind the tool to your Chat Model so Phi-4 knows it exists
model_with_tools = chat_model.bind_tools([search_local_database])

In [ ]:
# Step 1: Create some fake company data
fake_company_docs = [
    Document(
        page_content="OmniCorp Vacation Policy: Employees receive 20 days of paid time off (PTO) annually. PTO requests must be submitted through the HR portal at least 2 weeks in advance. Unused PTO does not roll over to the next calendar year.",
        metadata={"source": "hr_vacation_policy.txt"}
    ),
    Document(
        page_content="OmniCorp IT Equipment Guide: All remote employees are eligible for a $500 home office stipend. Company laptops must use the global corporate VPN at all times. Password changes are mandatory every 90 days.",
        metadata={"source": "it_guidelines.txt"}
    ),
    Document(
        page_content="OmniCorp Remote Work Guidelines: Core working hours are 10:00 AM to 4:00 PM EST. Employees working from home must ensure a quiet environment and a minimum internet speed of 50 Mbps down / 10 Mbps up for video conferencing.",
        metadata={"source": "remote_work_policy.txt"}
    )
]

# Step 2: Split the documents into smaller chunks
# (Even though these are short, splitting is a standard habit for RAG pipelines)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = text_splitter.split_documents(fake_company_docs)

# Step 3: Initialize your Embedding Model (all-MiniLM-L6-v2)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Step 4: Create ChromaDB and save it locally
# This creates a folder named 'chroma_db_storage' in your current directory
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db_storage"
)

print(f"Successfully embedded and saved {len(chunks)} text chunks to ChromaDB!")

/tmp/ipykernel_571/923222552.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Successfully embedded and saved 6 text chunks to ChromaDB!


In [ ]:
# Create a retriever object
retriever = vector_store.as_retriever(search_kwargs={"k": 2}) # Get top 2 most relevant chunks

# Run a test search
query = "How many hours can I work for in a week"
results = retriever.invoke(query)

# Print the results
print(f"Query: {query}\n" + "="*40)
for i, doc in enumerate(results):
    print(f"Result {i+1} from {doc.metadata['source']}:")
    print(f"{doc.page_content}\n")

Query: How many hours can I work for in a week
Result 1 from remote_work_policy.txt:
OmniCorp Remote Work Guidelines: Core working hours are 10:00 AM to 4:00 PM EST. Employees working from home must ensure a quiet environment and a minimum internet speed of 50 Mbps down / 10 Mbps up

Result 2 from it_guidelines.txt:
mandatory every 90 days.



In [ ]:
import chromadb

# Point Chroma to the folder we created
client = chromadb.PersistentClient(path="./chroma_db_storage")

# Get our default collection (Chroma names it 'langchain' by default if not specified)
collection = client.get_collection("langchain")

# Peek at the first 2 items inside
db_contents = collection.peek(2)
print(db_contents) # This will print a dictionary containing the IDs, metadata, text, and the vector numbers!

collections = client.list_collections()
print(collections)

{'ids': ['f5e9e824-5b4d-4849-8287-f0414301996d', 'fd5234bd-a2ac-46a7-81cf-c0fe5fcb2937'], 'embeddings': array([[ 8.57645296e-04,  1.05544440e-01,  4.77143377e-02,
         2.00365745e-02,  3.62576582e-02, -6.57896101e-02,
         2.09498070e-02, -3.65660526e-02, -4.62068692e-02,
        -1.89388564e-04,  5.57829775e-02,  6.70384318e-02,
        -4.90595885e-02, -9.30690381e-04,  2.43575554e-02,
         1.52386483e-02, -1.44230593e-02, -9.23834927e-03,
         4.95782755e-02,  1.93667787e-04,  6.13869168e-02,
        -7.98236728e-02, -6.77611381e-02,  2.15790179e-02,
         1.83982160e-02,  8.61671288e-03,  2.84498874e-02,
         5.59682027e-02, -8.42645988e-02,  9.63800251e-02,
        -5.59402630e-02, -3.53386849e-02, -3.58351357e-02,
        -3.63291427e-02,  3.09714004e-02,  4.58093584e-02,
        -5.17572798e-02, -6.49368092e-02, -9.38818157e-02,
        -6.66803168e-03,  6.67779008e-03, -8.32899357e-04,
        -2.12879591e-02,  1.48293665e-02,  9.35794506e-03,
         3.

In [ ]:
pip install huggingface_hub transformers torch langchain langchain-huggingface langchain-text-splitters langchain-community chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 